# 03 — Tensor Network Structures & Graphical Notation

A **tensor network** is a collection of tensors whose indices are (partially) contracted.  
We use a graphical language to reason about these contractions without writing out every index.

---

## 1. Graphical Notation

**Rules:**
- Each tensor = a **node** (box or circle)
- Each index = a **leg** (edge sticking out)
- A **connected** edge between two nodes = shared (contracted) index
- A **dangling** edge = free (open) index in the result

```
Scalar:   ●           (no legs)
Vector:   ●─           (one leg)
Matrix:  ─●─           (two legs)
3-tensor: ─●─          (three legs, e.g. one up, two horizontal)
           │

Matrix product A @ B:
  ─[A]─●─[B]─          (middle edge contracted, outer edges free)
```

The power: **any** multi-index summation (einsum) has a diagram.

In [ ]:
import numpy as np
import tensornetwork as tn
import matplotlib.pyplot as plt

# ── Simple 2-node contraction: inner product of two vectors ──
a = tn.Node(np.array([1.0, 2.0, 3.0]), name="a")
b = tn.Node(np.array([4.0, 5.0, 6.0]), name="b")

# Connect leg 0 of a to leg 0 of b (the single shared index)
edge = a[0] ^ b[0]   # ^ is short for tn.connect()

result = tn.contract(edge)   # sum over the connected index
print(f"Inner product a·b = {result.tensor}")   # should be 1*4+2*5+3*6 = 32

# Verify
print(f"numpy dot: {np.dot([1,2,3],[4,5,6])}")

In [ ]:
# ── Matrix multiplication as a 2-node contraction ──
A_data = np.random.randn(3, 4)
B_data = np.random.randn(4, 5)

A = tn.Node(A_data, name="A")
B = tn.Node(B_data, name="B")

# A has legs [0=row, 1=col], B has legs [0=row, 1=col]
# Contract: A's col index (leg 1) with B's row index (leg 0)
shared_edge = A[1] ^ B[0]
C_node = tn.contract(shared_edge)

print(f"A @ B shape: {C_node.tensor.shape}")   # (3, 5)
print(f"numpy A@B match: {np.allclose(C_node.tensor, A_data @ B_data)}")

## 2. The NCON Interface — Compact Contraction Specification

`ncon` (network contractor) lets you specify a whole network in one call:
- **Positive** index labels are contracted (summed over)
- **Negative** index labels are kept free

```
ncon([A, B], [[-1, 1], [1, -2]])  →  C[-1, -2] = sum_1 A[-1, 1] * B[1, -2]
                                   = matrix product  A @ B
```

In [ ]:
from tensornetwork import ncon

A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
C = np.random.randn(5, 6)

# Chain A @ B @ C  — contract two edges in one call
# A: legs (-1=free row, 1=contracted with B)
# B: legs (1=contracted with A, 2=contracted with C)
# C: legs (2=contracted with B, -2=free col)
result = ncon([A, B, C], [[-1, 1], [1, 2], [2, -2]])
print(f"A@B@C shape: {result.shape}")
print(f"numpy match: {np.allclose(result, A @ B @ C)}")

# Trace of a matrix — contracting both legs of M
M = np.eye(4) * 3
trace = ncon([M], [[1, 1]])   # both indices are label 1 -> trace
print(f"\nTrace of 3*I_4 = {trace}  (expected 12)")

## 3. Matrix Product State (MPS) / Tensor Train

An MPS is a 1-D chain of tensors:

```
  s1      s2      s3      s4
  │       │       │       │
[G1]─r1─[G2]─r2─[G3]─r3─[G4]
```

Each core $G^{(k)}$ has shape $(r_{k-1}, s_k, r_k)$:
- $s_k$ = physical index (dimension of the data at site $k$)
- $r_k$ = bond index (virtual / auxiliary)

To get the value $T_{s_1 s_2 s_3 s_4}$, contract over all bond indices:

$$T_{s_1 s_2 s_3 s_4} = \sum_{r_1 r_2 r_3} G^{(1)}_{1,s_1,r_1} G^{(2)}_{r_1,s_2,r_2} G^{(3)}_{r_2,s_3,r_3} G^{(4)}_{r_3,s_4,1}$$

In [ ]:
# Build a 4-site MPS manually and contract it to a full tensor
np.random.seed(5)
physical_dims = [3, 4, 3, 4]   # s_k dimensions
bond_dims     = [1, 5, 5, 5, 1]  # r_k dimensions (boundary=1)

cores = []
for k in range(4):
    shape = (bond_dims[k], physical_dims[k], bond_dims[k+1])
    cores.append(np.random.randn(*shape))
    print(f"G^({k+1}) shape: {shape}")

# Contract the MPS chain to full tensor using einsum
# G1: (1, s1, r1) → squeeze to (s1, r1)
# then contract G2, G3, G4 one after another
result = cores[0][0, :, :]       # (s1, r1)
result = np.einsum('ir,rjs->ijs', # (s1,r1) x (r1,s2,r2) -> (s1,s2,r2)
    result.reshape(physical_dims[0], bond_dims[1]), cores[1]).reshape(
    physical_dims[0]*physical_dims[1], bond_dims[2])

# Use tensorly utility for clean comparison
import tensorly as tl
T_mps = tl.tt_to_tensor([tl.tensor(c) for c in cores])
print(f"\nFull tensor shape from MPS: {T_mps.shape}")
print(f"Expected: {tuple(physical_dims)}")

## 4. Tree Tensor Networks (TTN)

Instead of a linear chain, tensors are arranged in a **binary tree**:

```
       [root]
       /    \
    [AB]    [CD]
    / \     / \
  [A][B]  [C][D]
```

- Leaf nodes hold the physical data
- Internal nodes contract information from subtrees
- More expressive than MPS for 2-D or hierarchical data
- Bond dimension grows logarithmically with the number of leaves

In [ ]:
# TTN contraction: 4 leaf tensors contracted into a root scalar
np.random.seed(11)
d = 4   # physical dimension at each leaf
r = 3   # bond dimension

# Leaf tensors: each is a vector of size d
leaves = [np.random.randn(d) for _ in range(4)]

# Level-1: pair-wise contraction tensors  shape (d, d, r)
W_left  = np.random.randn(d, d, r)   # contracts leaves[0] and leaves[1]
W_right = np.random.randn(d, d, r)   # contracts leaves[2] and leaves[3]

# Root tensor: shape (r, r)
W_root = np.random.randn(r, r)

# Contract level 1
v_left  = np.einsum('i,j,ijk->k', leaves[0], leaves[1], W_left)   # (r,)
v_right = np.einsum('i,j,ijk->k', leaves[2], leaves[3], W_right)  # (r,)

# Contract root
result_ttn = np.einsum('i,ij,j->', v_left, W_root, v_right)   # scalar

print(f"v_left shape:  {v_left.shape}")
print(f"v_right shape: {v_right.shape}")
print(f"TTN output (scalar): {result_ttn:.5f}")

## 5. Splitting Nodes — SVD on a Tensor Network Edge

A key operation in tensor networks is **SVD-based splitting**: take one big tensor and split it into two smaller tensors connected by a bond.

This is used in DMRG-style optimisation and in compressing MPS bonds.

In [ ]:
# Split a rank-4 tensor into two rank-3 tensors via SVD truncation
np.random.seed(42)
big = np.random.randn(6, 7, 8, 9)  # 4-way tensor

node = tn.Node(big, name="big")
# Split: left side keeps legs 0,1 ; right side keeps legs 2,3
# Bond (singular value) between them is truncated to max_singular_values=5
u, s, vh, trun_err = tn.split_node_full_svd(
    node,
    left_edges  = [node[0], node[1]],
    right_edges = [node[2], node[3]],
    max_singular_values=5
)

print(f"Original shape: {big.shape}")
print(f"U node shape:   {u.tensor.shape}")
print(f"S (sing vals):  {s.tensor.shape}  values={s.tensor.round(2)}")
print(f"Vh node shape:  {vh.tensor.shape}")
print(f"Truncation err: {trun_err.numpy():.6f}")

# Reconstruct and check error
big_recon = np.einsum('ija,a,akl->ijkl',
    u.tensor, s.tensor, vh.tensor)
print(f"Relative recon error: {norm(big - big_recon) / norm(big):.5f}")

## 6. Contraction Order Matters!

For a network of tensors, **the order in which you contract pairs** greatly affects computational cost.  
Optimal contraction ordering is related to graph theory / hypergraph partitioning.

`opt_einsum` automatically finds the best contraction path.

In [ ]:
import opt_einsum as oe

# A chain of 5 matrices: A1 @ A2 @ A3 @ A4 @ A5
dims = [(10, 100), (100, 5), (5, 200), (200, 8), (8, 50)]
mats = [np.random.randn(*d) for d in dims]

# Build einsum string: 'ij,jk,kl,lm,mn->in'
idx = 'abcdef'
einsum_str = ','.join(f'{idx[i]}{idx[i+1]}' for i in range(5))
einsum_str += f'->{idx[0]}{idx[5]}'
print("einsum string:", einsum_str)

# Find optimal path
path, info = oe.contract_path(einsum_str, *mats)
print("Optimal path:", path)
print(info)

# Compute result
result = oe.contract(einsum_str, *mats)
print(f"Result shape: {result.shape}")

## Summary

| Structure | Shape | Characteristic |
|-----------|-------|----------------|
| MPS / Tensor Train | 1-D chain | Efficient for 1-D correlated systems |
| TTN | Binary tree | Hierarchical, more expressive |
| MERA | Multi-scale tree | Long-range correlations, critical systems |
| PEPS | 2-D grid | 2-D systems (hard to contract exactly) |

Key operations:
- **Contraction**: sum over shared indices
- **SVD splitting**: decompose a node into two, introduce a bond
- **Truncation**: keep only top-r singular values to control cost

➡️ **04_ml_applications.ipynb** — using tensor networks in machine learning.